# 02 Exploratory Analysis

Exploratory business analysis of the expanded synthetic ecommerce dataset across customers, transactions, product catalog records, and daily product-event aggregates.


In [16]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

try:
    import plotly.express as px
    PLOTLY_AVAILABLE = True
except ModuleNotFoundError:
    px = None
    PLOTLY_AVAILABLE = False

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "raw" / "customers.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CUSTOMERS_PATH = PROJECT_ROOT / "data" / "raw" / "customers.csv"
TRANSACTIONS_PATH = PROJECT_ROOT / "data" / "raw" / "transactions.csv"
PRODUCTS_PATH = PROJECT_ROOT / "data" / "raw" / "products.csv"
PRODUCT_EVENTS_PATH = PROJECT_ROOT / "data" / "raw" / "product_events.csv"

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:,.3f}".format)

PLOTLY_AVAILABLE


False

In [17]:
def safe_divide(numerator, denominator):
    """Return a stable ratio while avoiding division by zero."""
    return np.where(denominator == 0, np.nan, numerator / denominator)


def format_currency(value):
    return f"${value:,.0f}"


def format_percent(value):
    return f"{value:.1%}"


def show_note(text):
    """Display a short business interpretation below an analysis block."""
    display(Markdown(f"**Interpretation:** {text}"))


def display_bar_table(frame, subset, color="#4C78A8", formats=None):
    """Display a compact table with in-cell bars as a fallback visualization."""
    styled = frame.style.format(formats or {})
    for column in subset:
        styled = styled.bar(subset=[column], color=color)
    display(styled)


def show_line(frame, x, y, title, y_label=None, color=None):
    """Show a Plotly line chart when available, otherwise show the underlying values."""
    if PLOTLY_AVAILABLE:
        fig = px.line(frame, x=x, y=y, color=color, markers=True, title=title, template="plotly_white")
        fig.update_layout(xaxis_title="", yaxis_title=y_label or y.replace("_", " ").title())
        fig.show()
    else:
        display(frame[[x, y]] if color is None else frame[[x, color, y]])


def show_multi_line(frame, x, value_columns, title, y_label="Value"):
    """Show multiple metrics over time in one chart."""
    plot_frame = frame[[x] + value_columns].melt(id_vars=x, var_name="metric", value_name="value")
    if PLOTLY_AVAILABLE:
        fig = px.line(plot_frame, x=x, y="value", color="metric", markers=True, title=title, template="plotly_white")
        fig.update_layout(xaxis_title="", yaxis_title=y_label, legend_title_text="Metric")
        fig.show()
    else:
        display(plot_frame)


def show_bar(frame, x, y, title, color=None, y_label=None, orientation="v"):
    """Show a Plotly bar chart when available, otherwise show a bar-styled table."""
    if PLOTLY_AVAILABLE:
        if orientation == "h":
            fig = px.bar(frame, x=y, y=x, color=color, orientation="h", title=title, template="plotly_white")
            fig.update_layout(xaxis_title=y_label or y.replace("_", " ").title(), yaxis_title="", showlegend=color is not None)
        else:
            fig = px.bar(frame, x=x, y=y, color=color, title=title, template="plotly_white")
            fig.update_layout(xaxis_title="", yaxis_title=y_label or y.replace("_", " ").title(), showlegend=color is not None)
        fig.show()
    else:
        columns = [x, y] if color is None else [x, color, y]
        display_bar_table(frame[columns], subset=[y], color="#59A14F")


def show_scatter(frame, x, y, title, color=None, size=None):
    """Show a Plotly scatter chart when available, otherwise show the underlying table."""
    if PLOTLY_AVAILABLE:
        fig = px.scatter(frame, x=x, y=y, color=color, size=size, hover_name="product_name" if "product_name" in frame.columns else None, title=title, template="plotly_white")
        fig.update_layout(xaxis_title=x.replace("_", " ").title(), yaxis_title=y.replace("_", " ").title())
        fig.show()
    else:
        display(frame)


## Load Expanded Synthetic Data


In [18]:
customers = pd.read_csv(
    CUSTOMERS_PATH,
    parse_dates=["first_purchase_date", "last_transaction_date"],
)
transactions = pd.read_csv(
    TRANSACTIONS_PATH,
    parse_dates=["transaction_date"],
)
products = pd.read_csv(PRODUCTS_PATH)
product_events = pd.read_csv(PRODUCT_EVENTS_PATH, parse_dates=["event_date"])

def normalize_bool(frame, columns):
    for column in columns:
        if column in frame.columns and not pd.api.types.is_bool_dtype(frame[column]):
            frame[column] = (
                frame[column]
                .astype(str)
                .str.lower()
                .map({"true": True, "false": False, "1": True, "0": False})
            )
    return frame

customers = normalize_bool(customers, ["price_increase_occurred", "churned"])
transactions = normalize_bool(transactions, ["price_increase_occurred", "churned"])
products = normalize_bool(products, ["stockout_flag"])
product_events = normalize_bool(product_events, ["stockout_flag", "price_increase_occurred"])

transactions["transaction_month"] = transactions["transaction_date"].dt.to_period("M").dt.to_timestamp()
product_events["event_month"] = product_events["event_date"].dt.to_period("M").dt.to_timestamp()
transactions["gross_merchandise_value"] = transactions["list_price"] * transactions["quantity"]
transactions["gross_margin_rate"] = transactions["gross_margin"] / transactions["order_value"]

customers["tenure_bucket"] = pd.cut(
    customers["customer_tenure_days"],
    bins=[-1, 90, 180, 365, 540, 731],
    labels=["0-90 days", "91-180 days", "181-365 days", "366-540 days", "541-730 days"],
)
customers["prior_spending_quintile"] = pd.qcut(customers["prior_spending"], q=5, duplicates="drop")
customers["purchase_frequency_bucket"] = pd.cut(
    customers["purchase_frequency"],
    bins=[0, 1, 2, 3, 4, 5, 7, 10, np.inf],
    labels=["0-1", "1-2", "2-3", "3-4", "4-5", "5-7", "7-10", "10+"],
    include_lowest=True,
)
customers["discount_usage_segment"] = pd.cut(
    customers["average_discount_percent"],
    bins=[-0.01, 0, 5, 10, 15, 20, 30.01],
    labels=["No average discount", "0-5%", "5-10%", "10-15%", "15-20%", "20-30%"],
)
customers["discount_user"] = customers["average_discount_percent"] > 0
customers["price_exposure_segment"] = customers["price_increase_occurred"].map(
    {True: "Exposed to price increase", False: "Not exposed"}
)

display(customers.head())
display(transactions.head())
display(products.head())
display(product_events.head())


,customer_id,first_purchase_date,customer_region,acquisition_channel,customer_tenure_days,purchase_frequency,prior_spending,transaction_count,total_spending,average_discount_percent,price_increase_occurred,churned,last_transaction_date,tenure_bucket,prior_spending_quintile,purchase_frequency_bucket,discount_usage_segment,discount_user,price_exposure_segment
0,C000001,2025-10-16,uk,direct,76,8.000,103.980,2,103.980,0.000,False,True,2025-11-08,0-90 days,"(84.412, 145.97]",7-10,No average discount,False,Not exposed
1,C000002,2025-07-12,uk,influencer,172,4.247,120.830,2,120.830,12.500,False,False,2025-10-31,91-180 days,"(84.412, 145.97]",4-5,10-15%,True,Not exposed
2,C000003,2025-07-12,north_america,influencer,172,6.371,170.770,3,170.770,6.670,False,False,2025-12-25,91-180 days,"(145.97, 224.842]",5-7,5-10%,True,Not exposed
3,C000004,2024-08-08,rest_of_world,influencer,510,5.729,491.930,8,491.930,8.750,False,False,2025-12-13,366-540 days,"(343.452, 1224.89]",5-7,5-10%,True,Not exposed
4,C000005,2025-07-03,uk,affiliate,181,4.036,104.970,2,104.970,0.000,True,False,2025-07-29,181-365 days,"(84.412, 145.97]",4-5,No average discount,False,Exposed to price increase


,transaction_id,customer_id,transaction_date,product_id,product_name,product_category,product_price,unit_cost,list_price,selling_price,discount_amount,discount_percent,quantity,order_value,gross_margin,customer_region,acquisition_channel,customer_tenure_days,purchase_frequency,prior_spending,price_increase_occurred,churned,transaction_month,gross_merchandise_value,gross_margin_rate
0,T00000001,C000001,2025-10-16,P0034,Core Puffer Vest,outerwear,81.990,30.490,81.990,81.990,0.000,0.000,1,81.990,51.500,uk,direct,0,0.000,0.000,False,True,2025-10-01,81.990,0.628
1,T00000002,C000001,2025-11-08,P0037,Core Crew Socks,accessories,21.990,7.770,21.990,21.990,0.000,0.000,1,21.990,14.220,uk,direct,23,4.000,81.990,False,True,2025-11-01,21.990,0.647
2,T00000003,C000002,2025-07-12,P0020,Motion Zip Hoodie,hoodies,66.990,28.300,66.990,56.940,10.050,15.000,1,56.940,28.640,uk,influencer,0,0.000,0.000,False,False,2025-07-01,66.990,0.503
3,T00000004,C000002,2025-10-31,P0021,Lift Oversized Hoodie,hoodies,70.990,26.700,70.990,63.890,7.100,10.000,1,63.890,37.190,uk,influencer,111,3.291,56.940,False,False,2025-10-01,70.990,0.582
4,T00000005,C000003,2025-07-12,P0003,Motion Seamless Leggings,leggings,54.990,20.190,54.990,54.990,0.000,0.000,1,54.990,34.800,north_america,influencer,0,0.000,0.000,False,False,2025-07-01,54.990,0.633


,product_id,product_name,product_category,unit_cost,list_price,gross_margin,gross_margin_rate,total_revenue,total_units_sold,total_purchases,product_views,add_to_cart_events,checkout_started_events,inventory_level,stockout_flag
0,P0001,Apex Sculpt Leggings,leggings,18.220,52.990,"63,143.250",0.627,"100,767.550",2065,1507,195059,19763,7967,79,True
1,P0002,Core Flex Leggings,leggings,25.200,53.990,"38,464.440",0.492,"78,204.840",1577,1214,170860,16926,6741,408,False
2,P0003,Motion Seamless Leggings,leggings,20.190,54.990,"58,885.750",0.598,"98,538.910",1964,1464,205376,20769,8377,424,False
3,P0004,Lift Training Leggings,leggings,27.070,55.990,"53,752.430",0.471,"114,010.250",2226,1681,208741,20888,8335,89,True
4,P0005,Contour High-Rise Leggings,leggings,18.320,43.990,"47,886.320",0.548,"87,402.560",2157,1639,215734,21663,8665,178,True


,event_date,product_id,product_name,product_category,unit_cost,list_price,selling_price,discount_percent,product_views,add_to_cart_events,checkout_started_events,purchases,units_sold,revenue,gross_margin,inventory_level,stockout_flag,price_increase_occurred,event_month
0,2024-01-01,P0001,Apex Sculpt Leggings,leggings,18.220,52.990,45.040,15.000,184,22,8,0,0,0.000,0.000,759,False,False,2024-01-01
1,2024-01-02,P0001,Apex Sculpt Leggings,leggings,18.220,52.990,39.740,25.000,233,37,16,0,0,0.000,0.000,759,False,False,2024-01-01
2,2024-01-03,P0001,Apex Sculpt Leggings,leggings,18.220,52.990,52.990,0.000,211,16,6,1,1,52.990,34.770,758,False,False,2024-01-01
3,2024-01-04,P0001,Apex Sculpt Leggings,leggings,18.220,52.990,47.690,10.000,217,21,8,1,2,95.380,58.940,756,False,False,2024-01-01
4,2024-01-05,P0001,Apex Sculpt Leggings,leggings,18.220,52.990,52.990,0.000,257,21,7,0,0,0.000,0.000,756,False,False,2024-01-01


## Dataset Overview


In [19]:
overview = pd.DataFrame(
    {
        "dataset": ["customers", "transactions", "products", "product_events"],
        "rows": [len(customers), len(transactions), len(products), len(product_events)],
        "columns": [customers.shape[1], transactions.shape[1], products.shape[1], product_events.shape[1]],
        "date_min": [
            customers["first_purchase_date"].min(),
            transactions["transaction_date"].min(),
            pd.NaT,
            product_events["event_date"].min(),
        ],
        "date_max": [
            customers["last_transaction_date"].max(),
            transactions["transaction_date"].max(),
            pd.NaT,
            product_events["event_date"].max(),
        ],
    }
)

display(overview)
display(transactions[["gross_merchandise_value", "order_value", "gross_margin", "discount_percent", "quantity"]].describe().T)
display(customers[["customer_tenure_days", "purchase_frequency", "prior_spending", "transaction_count", "total_spending"]].describe().T)
display(products[["list_price", "unit_cost", "gross_margin_rate", "total_revenue", "total_units_sold", "product_views", "inventory_level"]].describe().T)

show_note(
    f"The expanded raw layer links {len(customers):,} synthetic customers, {len(transactions):,} transactions, "
    f"{len(products):,} catalog products, and {len(product_events):,} daily product-event rows."
)


,dataset,rows,columns,date_min,date_max
0,customers,12500,19,2024-01-01,2025-12-31
1,transactions,48769,25,2024-01-01,2025-12-31
2,products,41,15,NaT,NaT
3,product_events,29971,19,2024-01-01,2025-12-31


,count,mean,std,min,25%,50%,75%,max
gross_merchandise_value,"48,769.000",63.057,39.860,17.990,35.990,54.990,71.980,403.960
order_value,"48,769.000",57.532,36.169,12.590,33.590,48.990,67.980,379.960
gross_margin,"48,769.000",30.672,20.485,4.520,17.740,25.600,37.260,226.480
discount_percent,"48,769.000",8.283,9.242,0.000,0.000,10.000,15.000,30.000
quantity,"48,769.000",1.329,0.629,1.000,1.000,1.000,2.000,4.000


,count,mean,std,min,25%,50%,75%,max
customer_tenure_days,"12,500.000",379.345,203.984,30.000,201.000,377.000,560.000,730.000
purchase_frequency,"12,500.000",4.151,2.234,0.502,2.509,3.829,5.345,16.000
prior_spending,"12,500.000",224.463,167.139,14.390,98.980,183.505,304.952,"1,224.890"
transaction_count,"12,500.000",3.902,2.633,1.000,2.000,3.000,5.000,18.000
total_spending,"12,500.000",224.463,167.139,14.390,98.980,183.505,304.952,"1,224.890"


,count,mean,std,min,25%,50%,75%,max
list_price,41.000,47.502,19.326,17.990,33.990,41.990,58.990,93.990
unit_cost,41.000,20.458,8.745,7.290,14.390,17.720,25.980,45.190
gross_margin_rate,41.000,0.529,0.061,0.440,0.471,0.520,0.590,0.627
total_revenue,41.000,"68,433.951","26,745.705","18,715.370","49,188.000","71,492.050","88,715.940","119,532.030"
total_units_sold,41.000,"1,581.341",313.614,"1,030.000","1,353.000","1,554.000","1,797.000","2,226.000"
product_views,41.000,"166,050.390","25,126.036","120,900.000","149,640.000","162,033.000","184,398.000","215,734.000"
inventory_level,41.000,306.220,236.629,30.000,118.000,180.000,426.000,817.000


**Interpretation:** The expanded raw layer links 12,500 synthetic customers, 48,769 transactions, 41 catalog products, and 29,971 daily product-event rows.

## GMV, Revenue, Gross Margin, and AOV Trends


In [20]:
monthly_summary = (
    transactions.groupby("transaction_month")
    .agg(
        gmv=("gross_merchandise_value", "sum"),
        revenue=("order_value", "sum"),
        gross_margin=("gross_margin", "sum"),
        transaction_volume=("transaction_id", "count"),
        units_sold=("quantity", "sum"),
        average_order_value=("order_value", "mean"),
        average_discount_percent=("discount_percent", "mean"),
        price_change_frequency=("price_increase_occurred", "mean"),
    )
    .reset_index()
)
monthly_summary["gross_margin_rate"] = monthly_summary["gross_margin"] / monthly_summary["revenue"]
monthly_summary["gmv_to_revenue_discount_rate"] = 1 - (monthly_summary["revenue"] / monthly_summary["gmv"])
monthly_summary["transaction_month_label"] = monthly_summary["transaction_month"].dt.strftime("%Y-%m")

display(monthly_summary)
show_multi_line(monthly_summary, "transaction_month", ["gmv", "revenue", "gross_margin"], "Monthly GMV, Revenue, and Gross Margin", "Dollars")
show_line(monthly_summary, "transaction_month", "transaction_volume", "Monthly Transaction Volume", "Transactions")
show_multi_line(monthly_summary, "transaction_month", ["average_order_value", "gross_margin_rate", "gmv_to_revenue_discount_rate"], "Monthly AOV, Margin Rate, and Discount Rate", "Value")

peak_revenue_month = monthly_summary.loc[monthly_summary["revenue"].idxmax()]
peak_margin_month = monthly_summary.loc[monthly_summary["gross_margin"].idxmax()]
show_note(
    f"Revenue peaks in {peak_revenue_month['transaction_month_label']} at {format_currency(peak_revenue_month['revenue'])}; "
    f"gross margin peaks in {peak_margin_month['transaction_month_label']} at {format_currency(peak_margin_month['gross_margin'])}. "
    f"Average monthly AOV is ${monthly_summary['average_order_value'].mean():,.2f}."
)


,transaction_month,gmv,revenue,gross_margin,transaction_volume,units_sold,average_order_value,average_discount_percent,price_change_frequency,gross_margin_rate,gmv_to_revenue_discount_rate,transaction_month_label
0,2024-01-01,"36,921.200","33,915.250","18,251.900",568,780,59.710,7.174,0.037,0.538,0.081,2024-01
1,2024-02-01,"39,554.860","36,284.530","19,360.620",637,814,56.962,7.527,0.074,0.534,0.083,2024-02
2,2024-03-01,"53,185.040","49,033.960","26,469.760",825,1096,59.435,7.273,0.088,0.540,0.078,2024-03
3,2024-04-01,"57,405.730","52,423.270","28,014.160",924,1227,56.735,7.863,0.078,0.534,0.087,2024-04
4,2024-05-01,"66,711.540","61,451.070","32,977.740",1093,1446,56.222,7.626,0.116,0.537,0.079,2024-05
5,2024-06-01,"72,796.790","66,645.070","35,724.150",1144,1521,58.256,8.059,0.123,0.536,0.085,2024-06
6,2024-07-01,"86,376.820","78,912.800","42,297.150",1357,1818,58.152,8.091,0.112,0.536,0.086,2024-07
7,2024-08-01,"86,981.630","80,104.510","43,217.910",1376,1837,58.215,7.445,0.140,0.540,0.079,2024-08
8,2024-09-01,"96,038.420","87,930.320","46,630.520",1554,2058,56.583,7.999,0.015,0.530,0.084,2024-09
9,2024-10-01,"108,933.590","100,241.720","53,533.730",1711,2241,58.587,7.396,0.008,0.534,0.080,2024-10


,transaction_month,metric,value
0,2024-01-01,gmv,"36,921.200"
1,2024-02-01,gmv,"39,554.860"
2,2024-03-01,gmv,"53,185.040"
3,2024-04-01,gmv,"57,405.730"
4,2024-05-01,gmv,"66,711.540"
...,...,...,...
67,2025-08-01,gross_margin,"90,260.000"
68,2025-09-01,gross_margin,"95,596.140"
69,2025-10-01,gross_margin,"100,425.790"
70,2025-11-01,gross_margin,"117,514.510"


,transaction_month,transaction_volume
0,2024-01-01,568
1,2024-02-01,637
2,2024-03-01,825
3,2024-04-01,924
4,2024-05-01,1093
5,2024-06-01,1144
6,2024-07-01,1357
7,2024-08-01,1376
8,2024-09-01,1554
9,2024-10-01,1711


,transaction_month,metric,value
0,2024-01-01,average_order_value,59.710
1,2024-02-01,average_order_value,56.962
2,2024-03-01,average_order_value,59.435
3,2024-04-01,average_order_value,56.735
4,2024-05-01,average_order_value,56.222
...,...,...,...
67,2025-08-01,gmv_to_revenue_discount_rate,0.085
68,2025-09-01,gmv_to_revenue_discount_rate,0.083
69,2025-10-01,gmv_to_revenue_discount_rate,0.078
70,2025-11-01,gmv_to_revenue_discount_rate,0.096


**Interpretation:** Revenue peaks in 2025-11 at $222,187; gross margin peaks in 2025-11 at $117,515. Average monthly AOV is $57.55.

## Conversion Funnel Performance


In [21]:
funnel_totals = pd.Series(
    {
        "product_views": product_events["product_views"].sum(),
        "add_to_cart_events": product_events["add_to_cart_events"].sum(),
        "checkout_started_events": product_events["checkout_started_events"].sum(),
        "purchases": product_events["purchases"].sum(),
    }
)

funnel_rates = pd.DataFrame(
    [
        {
            "stage": "Add to cart",
            "numerator": "add_to_cart_events",
            "denominator": "product_views",
            "rate": funnel_totals["add_to_cart_events"] / funnel_totals["product_views"],
        },
        {
            "stage": "Checkout started",
            "numerator": "checkout_started_events",
            "denominator": "add_to_cart_events",
            "rate": funnel_totals["checkout_started_events"] / funnel_totals["add_to_cart_events"],
        },
        {
            "stage": "Purchase after checkout",
            "numerator": "purchases",
            "denominator": "checkout_started_events",
            "rate": funnel_totals["purchases"] / funnel_totals["checkout_started_events"],
        },
        {
            "stage": "View to purchase",
            "numerator": "purchases",
            "denominator": "product_views",
            "rate": funnel_totals["purchases"] / funnel_totals["product_views"],
        },
    ]
)
funnel_rates["numerator_value"] = funnel_rates["numerator"].map(funnel_totals)
funnel_rates["denominator_value"] = funnel_rates["denominator"].map(funnel_totals)

monthly_funnel = (
    product_events.groupby("event_month")
    .agg(
        product_views=("product_views", "sum"),
        add_to_cart_events=("add_to_cart_events", "sum"),
        checkout_started_events=("checkout_started_events", "sum"),
        purchases=("purchases", "sum"),
    )
    .reset_index()
)
monthly_funnel["add_to_cart_rate"] = monthly_funnel["add_to_cart_events"] / monthly_funnel["product_views"]
monthly_funnel["checkout_start_rate"] = monthly_funnel["checkout_started_events"] / monthly_funnel["add_to_cart_events"]
monthly_funnel["purchase_after_checkout_rate"] = monthly_funnel["purchases"] / monthly_funnel["checkout_started_events"]
monthly_funnel["view_to_purchase_rate"] = monthly_funnel["purchases"] / monthly_funnel["product_views"]

display(pd.DataFrame(funnel_totals, columns=["total"]))
display(funnel_rates)
display(monthly_funnel)
show_bar(funnel_rates, "stage", "rate", "Overall Conversion Funnel Rates", y_label="Rate")
show_multi_line(monthly_funnel, "event_month", ["add_to_cart_rate", "checkout_start_rate", "purchase_after_checkout_rate", "view_to_purchase_rate"], "Monthly Funnel Rates", "Rate")

show_note(
    f"The sequential funnel converts {format_percent(funnel_rates.loc[funnel_rates['stage'] == 'Add to cart', 'rate'].iloc[0])} of views to carts, "
    f"{format_percent(funnel_rates.loc[funnel_rates['stage'] == 'Checkout started', 'rate'].iloc[0])} of carts to checkout, and "
    f"{format_percent(funnel_rates.loc[funnel_rates['stage'] == 'Purchase after checkout', 'rate'].iloc[0])} of checkout starts to purchases."
)


,total
product_views,6808066
add_to_cart_events,678053
checkout_started_events,270968
purchases,48769


,stage,numerator,denominator,rate,numerator_value,denominator_value
0,Add to cart,add_to_cart_events,product_views,0.100,678053,6808066
1,Checkout started,checkout_started_events,add_to_cart_events,0.400,270968,678053
2,Purchase after checkout,purchases,checkout_started_events,0.180,48769,270968
3,View to purchase,purchases,product_views,0.007,48769,6808066


,event_month,product_views,add_to_cart_events,checkout_started_events,purchases,add_to_cart_rate,checkout_start_rate,purchase_after_checkout_rate,view_to_purchase_rate
0,2024-01-01,235700,21784,8600,568,0.092,0.395,0.066,0.002
1,2024-02-01,222011,21028,8343,637,0.095,0.397,0.076,0.003
2,2024-03-01,259513,24706,9832,825,0.095,0.398,0.084,0.003
3,2024-04-01,253161,24284,9652,924,0.096,0.397,0.096,0.004
4,2024-05-01,264126,25557,10157,1093,0.097,0.397,0.108,0.004
5,2024-06-01,273222,26655,10634,1144,0.098,0.399,0.108,0.004
6,2024-07-01,285980,28124,11201,1357,0.098,0.398,0.121,0.005
7,2024-08-01,266888,25743,10214,1376,0.096,0.397,0.135,0.005
8,2024-09-01,262771,26058,10432,1554,0.099,0.400,0.149,0.006
9,2024-10-01,272980,26655,10651,1711,0.098,0.400,0.161,0.006


,stage,rate
0,Add to cart,0.099596
1,Checkout started,0.399627
2,Purchase after checkout,0.179981
3,View to purchase,0.007163


,event_month,metric,value
0,2024-01-01,add_to_cart_rate,0.092
1,2024-02-01,add_to_cart_rate,0.095
2,2024-03-01,add_to_cart_rate,0.095
3,2024-04-01,add_to_cart_rate,0.096
4,2024-05-01,add_to_cart_rate,0.097
...,...,...,...
91,2025-08-01,view_to_purchase_rate,0.010
92,2025-09-01,view_to_purchase_rate,0.010
93,2025-10-01,view_to_purchase_rate,0.010
94,2025-11-01,view_to_purchase_rate,0.011


**Interpretation:** The sequential funnel converts 10.0% of views to carts, 40.0% of carts to checkout, and 18.0% of checkout starts to purchases.

## Category-Level Performance


In [22]:
category_events = (
    product_events.groupby("product_category")
    .agg(
        product_days=("product_id", "count"),
        product_views=("product_views", "sum"),
        add_to_cart_events=("add_to_cart_events", "sum"),
        checkout_started_events=("checkout_started_events", "sum"),
        purchases=("purchases", "sum"),
        units_sold=("units_sold", "sum"),
        revenue=("revenue", "sum"),
        gross_margin=("gross_margin", "sum"),
        stockout_rate=("stockout_flag", "mean"),
        price_change_day_rate=("price_increase_occurred", "mean"),
        avg_selling_price=("selling_price", "mean"),
    )
)
category_transactions = transactions.groupby("product_category").agg(
    transactions=("transaction_id", "count"),
    customers=("customer_id", "nunique"),
    transaction_aov=("order_value", "mean"),
    avg_discount_percent=("discount_percent", "mean"),
)
category_products = products.groupby("product_category").agg(products=("product_id", "nunique"))

category_summary = (
    category_events.join(category_transactions, how="left")
    .join(category_products, how="left")
    .reset_index()
)
category_summary["gross_margin_rate"] = category_summary["gross_margin"] / category_summary["revenue"]
category_summary["add_to_cart_rate"] = category_summary["add_to_cart_events"] / category_summary["product_views"]
category_summary["checkout_start_rate"] = category_summary["checkout_started_events"] / category_summary["add_to_cart_events"]
category_summary["purchase_after_checkout_rate"] = category_summary["purchases"] / category_summary["checkout_started_events"]
category_summary["view_to_purchase_rate"] = category_summary["purchases"] / category_summary["product_views"]
category_summary["revenue_share"] = category_summary["revenue"] / category_summary["revenue"].sum()
category_summary = category_summary.sort_values("revenue", ascending=False)

display(category_summary)
show_bar(category_summary, "product_category", "revenue", "Revenue by Product Category", y_label="Revenue")
show_bar(category_summary, "product_category", "gross_margin", "Gross Margin by Product Category", y_label="Gross margin")
show_bar(category_summary.sort_values("stockout_rate", ascending=False), "product_category", "stockout_rate", "Stockout Rate by Category", y_label="Stockout rate")

leading_category = category_summary.iloc[0]
highest_margin_category = category_summary.sort_values("gross_margin_rate", ascending=False).iloc[0]
show_note(
    f"{leading_category['product_category']} leads revenue with {format_currency(leading_category['revenue'])} "
    f"({leading_category['revenue_share']:.1%} of product-event revenue). "
    f"{highest_margin_category['product_category']} has the strongest gross margin rate at {highest_margin_category['gross_margin_rate']:.1%}."
)


,product_category,product_days,product_views,add_to_cart_events,checkout_started_events,purchases,units_sold,revenue,gross_margin,stockout_rate,price_change_day_rate,avg_selling_price,transactions,customers,transaction_aov,avg_discount_percent,products,gross_margin_rate,add_to_cart_rate,checkout_start_rate,purchase_after_checkout_rate,view_to_purchase_rate,revenue_share
3,leggings,4386,1162534,116776,46792,8701,11595,"550,702.070","296,650.950",0.067,0.083,48.412,8701,6046,63.292,8.295,6,0.539,0.100,0.401,0.186,0.007,0.196
1,hoodies,3655,801218,79374,31662,5680,7449,"457,048.170","253,589.190",0.082,0.107,62.843,5680,4436,80.466,8.216,5,0.555,0.099,0.399,0.179,0.007,0.163
4,outerwear,2924,551959,54238,21663,3762,5039,"405,093.820","209,991.070",0.026,0.113,82.722,3762,3184,107.680,8.446,4,0.518,0.098,0.399,0.174,0.007,0.144
2,joggers,3655,823492,81853,32704,5857,7812,"404,130.040","215,115.580",0.105,0.108,52.501,5857,4512,68.999,8.178,5,0.532,0.099,0.400,0.179,0.007,0.144
7,training_tops,4386,1126385,112597,45027,8313,10996,"341,088.170","163,093.650",0.013,0.107,31.569,8313,5835,41.031,8.372,6,0.478,0.100,0.400,0.185,0.007,0.122
6,sports_bras,3655,902651,90083,35945,6679,8909,"306,074.200","174,897.650",0.045,0.131,35.155,6679,5027,45.826,8.108,5,0.571,0.100,0.399,0.186,0.007,0.109
5,shorts,3655,722515,71858,28724,4989,6636,"214,118.560","114,812.760",0.081,0.095,33.289,4989,3975,42.918,8.314,5,0.536,0.099,0.400,0.174,0.007,0.076
0,accessories,3655,717312,71274,28451,4788,6399,"127,536.980","67,672.880",0.032,0.108,20.317,4788,3863,26.637,8.401,5,0.531,0.099,0.399,0.168,0.007,0.045


,product_category,revenue
3,leggings,550702.070000
1,hoodies,457048.170000
4,outerwear,405093.820000
2,joggers,404130.040000
7,training_tops,341088.170000
6,sports_bras,306074.200000
5,shorts,214118.560000
0,accessories,127536.980000


,product_category,gross_margin
3,leggings,296650.950000
1,hoodies,253589.190000
4,outerwear,209991.070000
2,joggers,215115.580000
7,training_tops,163093.650000
6,sports_bras,174897.650000
5,shorts,114812.760000
0,accessories,67672.880000


,product_category,stockout_rate
2,joggers,0.104514
1,hoodies,0.082353
5,shorts,0.080711
3,leggings,0.067259
6,sports_bras,0.045417
0,accessories,0.031737
4,outerwear,0.026334
7,training_tops,0.012768


**Interpretation:** leggings leads revenue with $550,702 (19.6% of product-event revenue). sports_bras has the strongest gross margin rate at 57.1%.

## Product Revenue and Margin Leaders/Laggards


In [23]:
event_product_summary = product_events.groupby(["product_id", "product_name", "product_category"]).agg(
    event_days=("event_date", "nunique"),
    revenue=("revenue", "sum"),
    gross_margin=("gross_margin", "sum"),
    units_sold=("units_sold", "sum"),
    purchases=("purchases", "sum"),
    product_views=("product_views", "sum"),
    add_to_cart_events=("add_to_cart_events", "sum"),
    checkout_started_events=("checkout_started_events", "sum"),
    stockout_days=("stockout_flag", "sum"),
    price_change_days=("price_increase_occurred", "sum"),
    avg_selling_price=("selling_price", "mean"),
).reset_index()

product_performance = event_product_summary.copy()
product_performance["gross_margin_rate"] = product_performance["gross_margin"] / product_performance["revenue"]
product_performance["units_per_day"] = product_performance["units_sold"] / product_performance["event_days"]
product_performance["revenue_per_day"] = product_performance["revenue"] / product_performance["event_days"]
product_performance["stockout_rate"] = product_performance["stockout_days"] / product_performance["event_days"]
product_performance["price_change_day_rate"] = product_performance["price_change_days"] / product_performance["event_days"]
product_performance["view_to_purchase_rate"] = product_performance["purchases"] / product_performance["product_views"]

top_products_by_revenue = product_performance.nlargest(10, "revenue")
bottom_products_by_revenue = product_performance.nsmallest(10, "revenue")
top_products_by_margin = product_performance.nlargest(10, "gross_margin")
bottom_products_by_margin = product_performance.nsmallest(10, "gross_margin")

display(top_products_by_revenue)
display(bottom_products_by_revenue)
display(top_products_by_margin)
display(bottom_products_by_margin)
show_bar(top_products_by_revenue.sort_values("revenue"), "product_name", "revenue", "Top Products by Revenue", y_label="Revenue", orientation="h")
show_bar(bottom_products_by_revenue.sort_values("revenue", ascending=False), "product_name", "revenue", "Bottom Products by Revenue", y_label="Revenue", orientation="h")
show_bar(top_products_by_margin.sort_values("gross_margin"), "product_name", "gross_margin", "Top Products by Gross Margin", y_label="Gross margin", orientation="h")

top_revenue_product = top_products_by_revenue.iloc[0]
bottom_revenue_product = bottom_products_by_revenue.iloc[0]
top_margin_product = top_products_by_margin.iloc[0]
show_note(
    f"{top_revenue_product['product_name']} is the top revenue product at {format_currency(top_revenue_product['revenue'])}. "
    f"{top_margin_product['product_name']} contributes the most gross margin at {format_currency(top_margin_product['gross_margin'])}, "
    f"while {bottom_revenue_product['product_name']} is the lowest revenue product at {format_currency(bottom_revenue_product['revenue'])}."
)


,product_id,product_name,product_category,event_days,revenue,gross_margin,units_sold,purchases,product_views,add_to_cart_events,checkout_started_events,stockout_days,price_change_days,avg_selling_price,gross_margin_rate,units_per_day,revenue_per_day,stockout_rate,price_change_day_rate,view_to_purchase_rate
35,P0036,Lift Zip Jacket,outerwear,731,"119,532.030","67,849.320",1383,1049,149640,14763,5904,77,102,88.821,0.568,1.892,163.519,0.105,0.140,0.007
3,P0004,Lift Training Leggings,leggings,731,"114,010.250","53,752.430",2226,1681,208741,20888,8335,123,38,52.085,0.471,3.045,155.965,0.168,0.052,0.008
34,P0035,Motion Lightweight Jacket,outerwear,731,"104,910.410","47,383.540",1273,928,136353,13394,5334,0,98,85.293,0.452,1.741,143.516,0.000,0.134,0.007
33,P0034,Core Puffer Vest,outerwear,731,"100,938.350","59,685.380",1353,1016,145066,14358,5729,0,38,76.810,0.591,1.851,138.083,0.000,0.052,0.007
0,P0001,Apex Sculpt Leggings,leggings,731,"100,767.550","63,143.250",2065,1507,195059,19763,7967,135,117,49.538,0.627,2.825,137.849,0.185,0.160,0.008
2,P0003,Motion Seamless Leggings,leggings,731,"98,538.910","58,885.750",1964,1464,205376,20769,8377,0,49,51.246,0.598,2.687,134.800,0.000,0.067,0.007
19,P0020,Motion Zip Hoodie,hoodies,731,"98,421.580","53,226.480",1597,1186,158560,15716,6302,187,97,63.065,0.541,2.185,134.640,0.256,0.133,0.007
20,P0021,Lift Oversized Hoodie,hoodies,731,"98,200.870","57,910.570",1509,1148,166915,16570,6604,0,65,66.595,0.590,2.064,134.338,0.000,0.089,0.007
25,P0026,Lift Tapered Joggers,joggers,731,"97,414.600","45,912.580",1797,1311,181779,18119,7248,113,65,55.392,0.471,2.458,133.262,0.155,0.089,0.007
22,P0023,Apex Slim Joggers,joggers,731,"89,309.960","54,135.260",1710,1305,177049,17823,7103,0,112,53.501,0.606,2.339,122.175,0.000,0.153,0.007


,product_id,product_name,product_category,event_days,revenue,gross_margin,units_sold,purchases,product_views,add_to_cart_events,checkout_started_events,stockout_days,price_change_days,avg_selling_price,gross_margin_rate,units_per_day,revenue_per_day,stockout_rate,price_change_day_rate,view_to_purchase_rate
38,P0039,Motion Duffel Bag,accessories,731,"18,715.370","10,433.930",1136,840,131750,13071,5227,0,113,17.011,0.558,1.554,25.602,0.000,0.155,0.006
39,P0040,Lift Wrist Wraps,accessories,731,"21,339.460","9,915.090",1051,791,126883,12502,4979,5,107,20.823,0.465,1.438,29.192,0.007,0.146,0.006
36,P0037,Core Crew Socks,accessories,731,"26,591.170","16,311.460",1323,1002,150796,15026,6016,7,30,20.570,0.613,1.810,36.376,0.010,0.041,0.007
40,P0041,Tempo Water Bottle,accessories,731,"29,181.080","16,563.240",1447,1092,156746,15656,6248,100,45,20.583,0.568,1.979,39.919,0.137,0.062,0.007
37,P0038,Apex Training Cap,accessories,731,"31,709.900","14,449.160",1442,1063,151137,15019,5981,4,99,22.597,0.456,1.973,43.379,0.005,0.135,0.007
27,P0028,Apex Training Shorts,shorts,731,"37,864.490","17,753.080",1331,1025,139270,13848,5530,256,36,29.056,0.469,1.821,51.798,0.350,0.049,0.007
30,P0031,Lift Hybrid Shorts,shorts,731,"39,461.710","22,697.360",1165,861,129444,12808,5100,0,65,34.795,0.575,1.594,53.983,0.000,0.089,0.007
31,P0032,Tempo Woven Shorts,shorts,731,"40,976.010","21,465.120",1113,816,126674,12563,5022,0,121,37.893,0.524,1.523,56.055,0.000,0.166,0.006
29,P0030,Motion Bike Shorts,shorts,731,"44,847.760","26,886.850",1471,1117,161601,15953,6364,39,89,31.161,0.600,2.012,61.351,0.053,0.122,0.007
13,P0014,Motion Crop Tee,training_tops,731,"48,174.380","23,501.810",1543,1146,164277,16262,6472,45,49,31.909,0.488,2.111,65.902,0.062,0.067,0.007


,product_id,product_name,product_category,event_days,revenue,gross_margin,units_sold,purchases,product_views,add_to_cart_events,checkout_started_events,stockout_days,price_change_days,avg_selling_price,gross_margin_rate,units_per_day,revenue_per_day,stockout_rate,price_change_day_rate,view_to_purchase_rate
35,P0036,Lift Zip Jacket,outerwear,731,"119,532.030","67,849.320",1383,1049,149640,14763,5904,77,102,88.821,0.568,1.892,163.519,0.105,0.140,0.007
0,P0001,Apex Sculpt Leggings,leggings,731,"100,767.550","63,143.250",2065,1507,195059,19763,7967,135,117,49.538,0.627,2.825,137.849,0.185,0.160,0.008
33,P0034,Core Puffer Vest,outerwear,731,"100,938.350","59,685.380",1353,1016,145066,14358,5729,0,38,76.810,0.591,1.851,138.083,0.000,0.052,0.007
2,P0003,Motion Seamless Leggings,leggings,731,"98,538.910","58,885.750",1964,1464,205376,20769,8377,0,49,51.246,0.598,2.687,134.800,0.000,0.067,0.007
20,P0021,Lift Oversized Hoodie,hoodies,731,"98,200.870","57,910.570",1509,1148,166915,16570,6604,0,65,66.595,0.590,2.064,134.338,0.000,0.089,0.007
26,P0027,Tempo Woven Joggers,joggers,731,"87,124.690","54,491.290",1588,1192,156494,15545,6210,269,112,55.798,0.625,2.172,119.186,0.368,0.153,0.008
21,P0022,Tempo Fleece Hoodie,hoodies,731,"88,715.940","54,362.120",1438,1087,157067,15574,6197,0,47,62.970,0.613,1.967,121.362,0.000,0.064,0.007
22,P0023,Apex Slim Joggers,joggers,731,"89,309.960","54,135.260",1710,1305,177049,17823,7103,0,112,53.501,0.606,2.339,122.175,0.000,0.153,0.007
3,P0004,Lift Training Leggings,leggings,731,"114,010.250","53,752.430",2226,1681,208741,20888,8335,123,38,52.085,0.471,3.045,155.965,0.168,0.052,0.008
19,P0020,Motion Zip Hoodie,hoodies,731,"98,421.580","53,226.480",1597,1186,158560,15716,6302,187,97,63.065,0.541,2.185,134.640,0.256,0.133,0.007


,product_id,product_name,product_category,event_days,revenue,gross_margin,units_sold,purchases,product_views,add_to_cart_events,checkout_started_events,stockout_days,price_change_days,avg_selling_price,gross_margin_rate,units_per_day,revenue_per_day,stockout_rate,price_change_day_rate,view_to_purchase_rate
39,P0040,Lift Wrist Wraps,accessories,731,"21,339.460","9,915.090",1051,791,126883,12502,4979,5,107,20.823,0.465,1.438,29.192,0.007,0.146,0.006
38,P0039,Motion Duffel Bag,accessories,731,"18,715.370","10,433.930",1136,840,131750,13071,5227,0,113,17.011,0.558,1.554,25.602,0.000,0.155,0.006
37,P0038,Apex Training Cap,accessories,731,"31,709.900","14,449.160",1442,1063,151137,15019,5981,4,99,22.597,0.456,1.973,43.379,0.005,0.135,0.007
36,P0037,Core Crew Socks,accessories,731,"26,591.170","16,311.460",1323,1002,150796,15026,6016,7,30,20.570,0.613,1.810,36.376,0.010,0.041,0.007
40,P0041,Tempo Water Bottle,accessories,731,"29,181.080","16,563.240",1447,1092,156746,15656,6248,100,45,20.583,0.568,1.979,39.919,0.137,0.062,0.007
27,P0028,Apex Training Shorts,shorts,731,"37,864.490","17,753.080",1331,1025,139270,13848,5530,256,36,29.056,0.469,1.821,51.798,0.350,0.049,0.007
31,P0032,Tempo Woven Shorts,shorts,731,"40,976.010","21,465.120",1113,816,126674,12563,5022,0,121,37.893,0.524,1.523,56.055,0.000,0.166,0.006
12,P0013,Core Fitted Tank,training_tops,731,"49,289.900","21,698.820",1718,1308,183293,18012,7200,0,103,29.280,0.440,2.350,67.428,0.000,0.141,0.007
30,P0031,Lift Hybrid Shorts,shorts,731,"39,461.710","22,697.360",1165,861,129444,12808,5100,0,65,34.795,0.575,1.594,53.983,0.000,0.089,0.007
13,P0014,Motion Crop Tee,training_tops,731,"48,174.380","23,501.810",1543,1146,164277,16262,6472,45,49,31.909,0.488,2.111,65.902,0.062,0.067,0.007


,product_name,revenue
22,Apex Slim Joggers,89309.960000
25,Lift Tapered Joggers,97414.600000
20,Lift Oversized Hoodie,98200.870000
19,Motion Zip Hoodie,98421.580000
2,Motion Seamless Leggings,98538.910000
0,Apex Sculpt Leggings,100767.550000
33,Core Puffer Vest,100938.350000
34,Motion Lightweight Jacket,104910.410000
3,Lift Training Leggings,114010.250000
35,Lift Zip Jacket,119532.030000


,product_name,revenue
13,Motion Crop Tee,48174.380000
29,Motion Bike Shorts,44847.760000
31,Tempo Woven Shorts,40976.010000
30,Lift Hybrid Shorts,39461.710000
27,Apex Training Shorts,37864.490000
37,Apex Training Cap,31709.900000
40,Tempo Water Bottle,29181.080000
36,Core Crew Socks,26591.170000
39,Lift Wrist Wraps,21339.460000
38,Motion Duffel Bag,18715.370000


,product_name,gross_margin
19,Motion Zip Hoodie,53226.480000
3,Lift Training Leggings,53752.430000
22,Apex Slim Joggers,54135.260000
21,Tempo Fleece Hoodie,54362.120000
26,Tempo Woven Joggers,54491.290000
20,Lift Oversized Hoodie,57910.570000
2,Motion Seamless Leggings,58885.750000
33,Core Puffer Vest,59685.380000
0,Apex Sculpt Leggings,63143.250000
35,Lift Zip Jacket,67849.320000


**Interpretation:** Lift Zip Jacket is the top revenue product at $119,532. Lift Zip Jacket contributes the most gross margin at $67,849, while Motion Duffel Bag is the lowest revenue product at $18,715.

## Product Sales Velocity and Slow Movers


In [24]:
sales_velocity = product_performance.sort_values("units_per_day", ascending=False)
fastest_products = sales_velocity.head(10)
slowest_products = sales_velocity.tail(10).sort_values("units_per_day")

display(fastest_products[["product_id", "product_name", "product_category", "units_sold", "units_per_day", "revenue_per_day", "view_to_purchase_rate", "stockout_rate"]])
display(slowest_products[["product_id", "product_name", "product_category", "units_sold", "units_per_day", "revenue_per_day", "view_to_purchase_rate", "stockout_rate"]])
show_bar(fastest_products.sort_values("units_per_day"), "product_name", "units_per_day", "Fastest Products by Units Sold per Day", y_label="Units per day", orientation="h")
show_bar(slowest_products.sort_values("units_per_day", ascending=False), "product_name", "units_per_day", "Slowest Products by Units Sold per Day", y_label="Units per day", orientation="h")
show_scatter(product_performance, "product_views", "units_per_day", "Product Views vs Sales Velocity", color="product_category", size="revenue")

show_note(
    f"Sales velocity ranges from {slowest_products['units_per_day'].min():.2f} to {fastest_products['units_per_day'].max():.2f} units per product-day, "
    f"which gives the dashboard a useful basis for top-product and slow-mover views."
)


,product_id,product_name,product_category,units_sold,units_per_day,revenue_per_day,view_to_purchase_rate,stockout_rate
3,P0004,Lift Training Leggings,leggings,2226,3.045,155.965,0.008,0.168
15,P0016,Everyday Oversized Tee,training_tops,2160,2.955,97.800,0.008,0.000
4,P0005,Contour High-Rise Leggings,leggings,2157,2.951,119.566,0.008,0.051
0,P0001,Apex Sculpt Leggings,leggings,2065,2.825,137.849,0.008,0.185
6,P0007,Apex Support Sports Bra,sports_bras,2054,2.810,82.778,0.008,0.011
2,P0003,Motion Seamless Leggings,leggings,1964,2.687,134.800,0.007,0.000
11,P0012,Apex Training Tee,training_tops,1960,2.681,83.352,0.008,0.015
7,P0008,Core Strappy Sports Bra,sports_bras,1903,2.603,91.348,0.008,0.105
8,P0009,Motion Longline Sports Bra,sports_bras,1869,2.557,98.604,0.008,0.063
14,P0015,Lift Performance Top,training_tops,1821,2.491,75.305,0.007,0.000


,product_id,product_name,product_category,units_sold,units_per_day,revenue_per_day,view_to_purchase_rate,stockout_rate
32,P0033,Apex Training Jacket,outerwear,1030,1.409,109.047,0.006,0.000
39,P0040,Lift Wrist Wraps,accessories,1051,1.438,29.192,0.006,0.007
31,P0032,Tempo Woven Shorts,shorts,1113,1.523,56.055,0.006,0.000
38,P0039,Motion Duffel Bag,accessories,1136,1.554,25.602,0.006,0.000
30,P0031,Lift Hybrid Shorts,shorts,1165,1.594,53.983,0.007,0.000
34,P0035,Motion Lightweight Jacket,outerwear,1273,1.741,143.516,0.007,0.000
24,P0025,Motion Training Joggers,joggers,1299,1.777,80.372,0.007,0.000
36,P0037,Core Crew Socks,accessories,1323,1.810,36.376,0.007,0.010
27,P0028,Apex Training Shorts,shorts,1331,1.821,51.798,0.007,0.350
18,P0019,Core Pullover Hoodie,hoodies,1351,1.848,116.856,0.007,0.156


,product_name,units_per_day
14,Lift Performance Top,2.491108
8,Motion Longline Sports Bra,2.556772
7,Core Strappy Sports Bra,2.603283
11,Apex Training Tee,2.681259
2,Motion Seamless Leggings,2.686731
6,Apex Support Sports Bra,2.809850
0,Apex Sculpt Leggings,2.824897
4,Contour High-Rise Leggings,2.950752
15,Everyday Oversized Tee,2.954856
3,Lift Training Leggings,3.045144


,product_name,units_per_day
18,Core Pullover Hoodie,1.848153
27,Apex Training Shorts,1.820793
36,Core Crew Socks,1.809850
24,Motion Training Joggers,1.777018
34,Motion Lightweight Jacket,1.741450
30,Lift Hybrid Shorts,1.593707
38,Motion Duffel Bag,1.554036
31,Tempo Woven Shorts,1.522572
39,Lift Wrist Wraps,1.437756
32,Apex Training Jacket,1.409029


,product_id,product_name,product_category,event_days,revenue,gross_margin,units_sold,purchases,product_views,add_to_cart_events,checkout_started_events,stockout_days,price_change_days,avg_selling_price,gross_margin_rate,units_per_day,revenue_per_day,stockout_rate,price_change_day_rate,view_to_purchase_rate
0,P0001,Apex Sculpt Leggings,leggings,731,"100,767.550","63,143.250",2065,1507,195059,19763,7967,135,117,49.538,0.627,2.825,137.849,0.185,0.160,0.008
1,P0002,Core Flex Leggings,leggings,731,"78,204.840","38,464.440",1577,1214,170860,16926,6741,0,57,50.706,0.492,2.157,106.983,0.000,0.078,0.007
2,P0003,Motion Seamless Leggings,leggings,731,"98,538.910","58,885.750",1964,1464,205376,20769,8377,0,49,51.246,0.598,2.687,134.800,0.000,0.067,0.007
3,P0004,Lift Training Leggings,leggings,731,"114,010.250","53,752.430",2226,1681,208741,20888,8335,123,38,52.085,0.471,3.045,155.965,0.168,0.052,0.008
4,P0005,Contour High-Rise Leggings,leggings,731,"87,402.560","47,886.320",2157,1639,215734,21663,8665,37,60,41.197,0.548,2.951,119.566,0.051,0.082,0.008
5,P0006,Pulse Pocket Leggings,leggings,731,"71,777.960","34,518.760",1606,1196,166764,16767,6707,0,42,45.698,0.481,2.197,98.191,0.000,0.057,0.007
6,P0007,Apex Support Sports Bra,sports_bras,731,"60,510.930","31,364.670",2054,1535,204491,20675,8266,8,106,30.018,0.518,2.810,82.778,0.011,0.145,0.008
7,P0008,Core Strappy Sports Bra,sports_bras,731,"66,775.100","40,133.100",1903,1449,192575,19157,7642,77,106,35.710,0.601,2.603,91.348,0.105,0.145,0.008
8,P0009,Motion Longline Sports Bra,sports_bras,731,"72,079.750","45,222.220",1869,1401,184867,18519,7413,46,83,39.349,0.627,2.557,98.604,0.063,0.114,0.008
9,P0010,Lift High-Support Bra,sports_bras,731,"49,188.000","28,265.000",1525,1124,159786,15802,6277,0,87,33.037,0.575,2.086,67.289,0.000,0.119,0.007


**Interpretation:** Sales velocity ranges from 1.41 to 3.05 units per product-day, which gives the dashboard a useful basis for top-product and slow-mover views.

## Stockout Frequency


In [25]:
stockout_by_product = product_performance.sort_values("stockout_rate", ascending=False)
stockout_by_category = (
    product_events.groupby("product_category")
    .agg(
        product_days=("stockout_flag", "size"),
        stockout_days=("stockout_flag", "sum"),
        stockout_rate=("stockout_flag", "mean"),
        revenue=("revenue", "sum"),
        units_sold=("units_sold", "sum"),
    )
    .reset_index()
    .sort_values("stockout_rate", ascending=False)
)

display(stockout_by_product[["product_id", "product_name", "product_category", "stockout_days", "event_days", "stockout_rate", "units_per_day", "revenue"]].head(15))
display(stockout_by_category)
show_bar(stockout_by_product.head(10).sort_values("stockout_rate"), "product_name", "stockout_rate", "Products with Highest Stockout Frequency", y_label="Stockout rate", orientation="h")
show_bar(stockout_by_category, "product_category", "stockout_rate", "Stockout Frequency by Category", y_label="Stockout rate")

highest_stockout_product = stockout_by_product.iloc[0]
highest_stockout_category = stockout_by_category.iloc[0]
show_note(
    f"Overall product-day stockout frequency is {product_events['stockout_flag'].mean():.1%}. "
    f"{highest_stockout_product['product_name']} has the highest product-level stockout rate at {highest_stockout_product['stockout_rate']:.1%}, "
    f"and {highest_stockout_category['product_category']} is the highest-stockout category at {highest_stockout_category['stockout_rate']:.1%}."
)


,product_id,product_name,product_category,stockout_days,event_days,stockout_rate,units_per_day,revenue
26,P0027,Tempo Woven Joggers,joggers,269,731,0.368,2.172,"87,124.690"
27,P0028,Apex Training Shorts,shorts,256,731,0.350,1.821,"37,864.490"
19,P0020,Motion Zip Hoodie,hoodies,187,731,0.256,2.185,"98,421.580"
0,P0001,Apex Sculpt Leggings,leggings,135,731,0.185,2.825,"100,767.550"
3,P0004,Lift Training Leggings,leggings,123,731,0.168,3.045,"114,010.250"
18,P0019,Core Pullover Hoodie,hoodies,114,731,0.156,1.848,"85,422.080"
25,P0026,Lift Tapered Joggers,joggers,113,731,0.155,2.458,"97,414.600"
40,P0041,Tempo Water Bottle,accessories,100,731,0.137,1.979,"29,181.080"
35,P0036,Lift Zip Jacket,outerwear,77,731,0.105,1.892,"119,532.030"
7,P0008,Core Strappy Sports Bra,sports_bras,77,731,0.105,2.603,"66,775.100"


,product_category,product_days,stockout_days,stockout_rate,revenue,units_sold
2,joggers,3655,382,0.105,"404,130.040",7812
1,hoodies,3655,301,0.082,"457,048.170",7449
5,shorts,3655,295,0.081,"214,118.560",6636
3,leggings,4386,295,0.067,"550,702.070",11595
6,sports_bras,3655,166,0.045,"306,074.200",8909
0,accessories,3655,116,0.032,"127,536.980",6399
4,outerwear,2924,77,0.026,"405,093.820",5039
7,training_tops,4386,56,0.013,"341,088.170",10996


,product_name,stockout_rate
35,Lift Zip Jacket,0.105335
7,Core Strappy Sports Bra,0.105335
40,Tempo Water Bottle,0.136799
25,Lift Tapered Joggers,0.154583
18,Core Pullover Hoodie,0.155951
3,Lift Training Leggings,0.168263
0,Apex Sculpt Leggings,0.184679
19,Motion Zip Hoodie,0.255814
27,Apex Training Shorts,0.350205
26,Tempo Woven Joggers,0.367989


,product_category,stockout_rate
2,joggers,0.104514
1,hoodies,0.082353
5,shorts,0.080711
3,leggings,0.067259
6,sports_bras,0.045417
0,accessories,0.031737
4,outerwear,0.026334
7,training_tops,0.012768


**Interpretation:** Overall product-day stockout frequency is 5.6%. Tempo Woven Joggers has the highest product-level stockout rate at 36.8%, and joggers is the highest-stockout category at 10.5%.

## Product-Level Price Changes vs. Demand

This section is descriptive only. It compares product-event demand on price-increase days versus non-price-increase days; it does not estimate causal price elasticity.


In [26]:
product_price_periods = (
    product_events.groupby(["product_id", "product_name", "product_category", "price_increase_occurred"])
    .agg(
        days=("event_date", "nunique"),
        avg_selling_price=("selling_price", "mean"),
        product_views=("product_views", "sum"),
        purchases=("purchases", "sum"),
        units_sold=("units_sold", "sum"),
        revenue=("revenue", "sum"),
        gross_margin=("gross_margin", "sum"),
    )
    .reset_index()
)
product_price_periods["units_per_day"] = product_price_periods["units_sold"] / product_price_periods["days"]
product_price_periods["revenue_per_day"] = product_price_periods["revenue"] / product_price_periods["days"]
product_price_periods["view_to_purchase_rate"] = product_price_periods["purchases"] / product_price_periods["product_views"]

normal_periods = product_price_periods[~product_price_periods["price_increase_occurred"]].set_index("product_id")
increased_periods = product_price_periods[product_price_periods["price_increase_occurred"]].set_index("product_id")

price_demand_comparison = normal_periods.join(
    increased_periods,
    lsuffix="_normal",
    rsuffix="_price_increase",
    how="inner",
).reset_index()
price_demand_comparison["product_name"] = price_demand_comparison["product_name_normal"]
price_demand_comparison["product_category"] = price_demand_comparison["product_category_normal"]
price_demand_comparison["price_lift_pct"] = (
    price_demand_comparison["avg_selling_price_price_increase"]
    / price_demand_comparison["avg_selling_price_normal"]
    - 1
)
price_demand_comparison["unit_velocity_change_pct"] = (
    price_demand_comparison["units_per_day_price_increase"]
    / price_demand_comparison["units_per_day_normal"]
    - 1
)
price_demand_comparison["conversion_change_pp"] = (
    price_demand_comparison["view_to_purchase_rate_price_increase"]
    - price_demand_comparison["view_to_purchase_rate_normal"]
) * 100
price_demand_comparison["revenue_per_day_change_pct"] = (
    price_demand_comparison["revenue_per_day_price_increase"]
    / price_demand_comparison["revenue_per_day_normal"]
    - 1
)

price_period_summary = (
    product_events.groupby("price_increase_occurred")
    .agg(
        days=("event_date", "count"),
        avg_selling_price=("selling_price", "mean"),
        product_views=("product_views", "sum"),
        purchases=("purchases", "sum"),
        units_sold=("units_sold", "sum"),
        revenue=("revenue", "sum"),
        gross_margin=("gross_margin", "sum"),
    )
    .reset_index()
)
price_period_summary["units_per_day"] = price_period_summary["units_sold"] / price_period_summary["days"]
price_period_summary["revenue_per_day"] = price_period_summary["revenue"] / price_period_summary["days"]
price_period_summary["view_to_purchase_rate"] = price_period_summary["purchases"] / price_period_summary["product_views"]

display(price_period_summary)
display(
    price_demand_comparison[
        [
            "product_id",
            "product_name",
            "product_category",
            "price_lift_pct",
            "unit_velocity_change_pct",
            "conversion_change_pp",
            "revenue_per_day_change_pct",
            "days_normal",
            "days_price_increase",
        ]
    ].sort_values("unit_velocity_change_pct")
)
show_scatter(
    price_demand_comparison,
    "price_lift_pct",
    "unit_velocity_change_pct",
    "Product Price Lift vs Unit Velocity Change",
    color="product_category",
    size="revenue_price_increase",
)
show_scatter(
    price_demand_comparison,
    "price_lift_pct",
    "conversion_change_pp",
    "Product Price Lift vs Conversion Change",
    color="product_category",
    size="revenue_price_increase",
)

avg_price_lift = price_demand_comparison["price_lift_pct"].mean()
avg_velocity_change = price_demand_comparison["unit_velocity_change_pct"].mean()
avg_revenue_day_change = price_demand_comparison["revenue_per_day_change_pct"].mean()
show_note(
    f"Across products with both price-increase and normal days, average selling price is {avg_price_lift:.1%} higher on price-increase days; "
    f"unit velocity is {avg_velocity_change:.1%} different and revenue per product-day is {avg_revenue_day_change:.1%} different. "
    "This is descriptive demand movement, not a causal elasticity estimate."
)


,price_increase_occurred,days,avg_selling_price,product_views,purchases,units_sold,revenue,gross_margin,units_per_day,revenue_per_day,view_to_purchase_rate
0,False,26804,44.336,6122502,44466,59088,"2,543,306.220","1,348,574.270",2.204,94.885,0.007
1,True,3167,47.460,685564,4303,5747,"262,485.790","147,249.460",1.815,82.882,0.006


,product_id,product_name,product_category,price_lift_pct,unit_velocity_change_pct,conversion_change_pp,revenue_per_day_change_pct,days_normal,days_price_increase
5,P0006,Pulse Pocket Leggings,leggings,0.112,-0.772,-0.530,-0.757,689,42
3,P0004,Lift Training Leggings,leggings,0.099,-0.615,-0.507,-0.578,693,38
31,P0032,Tempo Woven Shorts,shorts,0.089,-0.578,-0.377,-0.551,610,121
2,P0003,Motion Seamless Leggings,leggings,0.076,-0.561,-0.407,-0.540,682,49
25,P0026,Lift Tapered Joggers,joggers,0.070,-0.460,-0.305,-0.432,666,65
21,P0022,Tempo Fleece Hoodie,hoodies,0.073,-0.454,-0.322,-0.413,684,47
6,P0007,Apex Support Sports Bra,sports_bras,0.086,-0.420,-0.232,-0.389,625,106
35,P0036,Lift Zip Jacket,outerwear,0.045,-0.419,-0.259,-0.405,629,102
38,P0039,Motion Duffel Bag,accessories,0.090,-0.402,-0.213,-0.355,618,113
10,P0011,Contour V-Neck Sports Bra,sports_bras,0.088,-0.400,-0.220,-0.343,634,97


,product_id,product_name_normal,product_category_normal,price_increase_occurred_normal,days_normal,avg_selling_price_normal,product_views_normal,purchases_normal,units_sold_normal,revenue_normal,gross_margin_normal,units_per_day_normal,revenue_per_day_normal,view_to_purchase_rate_normal,product_name_price_increase,product_category_price_increase,price_increase_occurred_price_increase,days_price_increase,avg_selling_price_price_increase,product_views_price_increase,purchases_price_increase,units_sold_price_increase,revenue_price_increase,gross_margin_price_increase,units_per_day_price_increase,revenue_per_day_price_increase,view_to_purchase_rate_price_increase,product_name,product_category,price_lift_pct,unit_velocity_change_pct,conversion_change_pp,revenue_per_day_change_pct
0,P0001,Apex Sculpt Leggings,leggings,False,614,49.202,164964,1258,1699,"82,162.490","51,206.710",2.767,133.815,0.008,Apex Sculpt Leggings,leggings,True,117,51.303,30095,249,366,"18,605.060","11,936.540",3.128,159.018,0.008,Apex Sculpt Leggings,leggings,0.043,0.130,0.065,0.188
1,P0002,Core Flex Leggings,leggings,False,674,50.473,156980,1088,1414,"69,759.570","34,126.770",2.098,103.501,0.007,Core Flex Leggings,leggings,True,57,53.452,13880,126,163,"8,445.270","4,337.670",2.860,148.163,0.009,Core Flex Leggings,leggings,0.059,0.363,0.215,0.432
2,P0003,Motion Seamless Leggings,leggings,False,682,50.986,192683,1422,1904,"95,387.130","56,945.370",2.792,139.864,0.007,Motion Seamless Leggings,leggings,True,49,54.867,12693,42,60,"3,151.780","1,940.380",1.224,64.322,0.003,Motion Seamless Leggings,leggings,0.076,-0.561,-0.407,-0.540
3,P0004,Lift Training Leggings,leggings,False,693,51.817,199118,1650,2180,"111,432.650","52,420.050",3.146,160.797,0.008,Lift Training Leggings,leggings,True,38,56.963,9623,31,46,"2,577.600","1,332.380",1.211,67.832,0.003,Lift Training Leggings,leggings,0.099,-0.615,-0.507,-0.578
4,P0005,Contour High-Rise Leggings,leggings,False,671,40.862,199009,1532,2010,"80,946.890","44,123.690",2.996,120.636,0.008,Contour High-Rise Leggings,leggings,True,60,44.952,16725,107,147,"6,455.670","3,762.630",2.450,107.594,0.006,Contour High-Rise Leggings,leggings,0.100,-0.182,-0.130,-0.108
5,P0006,Pulse Pocket Leggings,leggings,False,689,45.407,158776,1179,1584,"70,730.580","33,981.780",2.299,102.657,0.007,Pulse Pocket Leggings,leggings,True,42,50.471,7988,17,22,"1,047.380",536.980,0.524,24.938,0.002,Pulse Pocket Leggings,leggings,0.112,-0.772,-0.530,-0.757
6,P0007,Apex Support Sports Bra,sports_bras,False,625,29.648,178748,1394,1870,"54,828.150","28,292.850",2.992,87.725,0.008,Apex Support Sports Bra,sports_bras,True,106,32.200,25743,141,184,"5,682.780","3,071.820",1.736,53.611,0.005,Apex Support Sports Bra,sports_bras,0.086,-0.420,-0.232,-0.389
7,P0008,Core Strappy Sports Bra,sports_bras,False,625,35.319,166055,1253,1658,"57,682.940","34,470.940",2.653,92.293,0.008,Core Strappy Sports Bra,sports_bras,True,106,38.012,26520,196,245,"9,092.160","5,662.160",2.311,85.775,0.007,Core Strappy Sports Bra,sports_bras,0.076,-0.129,-0.016,-0.071
8,P0009,Motion Longline Sports Bra,sports_bras,False,648,39.057,165166,1299,1735,"66,695.800","41,763.850",2.677,102.926,0.008,Motion Longline Sports Bra,sports_bras,True,83,41.630,19701,102,134,"5,383.950","3,458.370",1.614,64.867,0.005,Motion Longline Sports Bra,sports_bras,0.066,-0.397,-0.269,-0.370
9,P0010,Lift High-Support Bra,sports_bras,False,644,32.740,142526,1042,1405,"45,069.700","25,793.100",2.182,69.984,0.007,Lift High-Support Bra,sports_bras,True,87,35.235,17260,82,120,"4,118.300","2,471.900",1.379,47.337,0.005,Lift High-Support Bra,sports_bras,0.076,-0.368,-0.256,-0.324


,product_id,product_name_normal,product_category_normal,price_increase_occurred_normal,days_normal,avg_selling_price_normal,product_views_normal,purchases_normal,units_sold_normal,revenue_normal,gross_margin_normal,units_per_day_normal,revenue_per_day_normal,view_to_purchase_rate_normal,product_name_price_increase,product_category_price_increase,price_increase_occurred_price_increase,days_price_increase,avg_selling_price_price_increase,product_views_price_increase,purchases_price_increase,units_sold_price_increase,revenue_price_increase,gross_margin_price_increase,units_per_day_price_increase,revenue_per_day_price_increase,view_to_purchase_rate_price_increase,product_name,product_category,price_lift_pct,unit_velocity_change_pct,conversion_change_pp,revenue_per_day_change_pct
0,P0001,Apex Sculpt Leggings,leggings,False,614,49.202,164964,1258,1699,"82,162.490","51,206.710",2.767,133.815,0.008,Apex Sculpt Leggings,leggings,True,117,51.303,30095,249,366,"18,605.060","11,936.540",3.128,159.018,0.008,Apex Sculpt Leggings,leggings,0.043,0.130,0.065,0.188
1,P0002,Core Flex Leggings,leggings,False,674,50.473,156980,1088,1414,"69,759.570","34,126.770",2.098,103.501,0.007,Core Flex Leggings,leggings,True,57,53.452,13880,126,163,"8,445.270","4,337.670",2.860,148.163,0.009,Core Flex Leggings,leggings,0.059,0.363,0.215,0.432
2,P0003,Motion Seamless Leggings,leggings,False,682,50.986,192683,1422,1904,"95,387.130","56,945.370",2.792,139.864,0.007,Motion Seamless Leggings,leggings,True,49,54.867,12693,42,60,"3,151.780","1,940.380",1.224,64.322,0.003,Motion Seamless Leggings,leggings,0.076,-0.561,-0.407,-0.540
3,P0004,Lift Training Leggings,leggings,False,693,51.817,199118,1650,2180,"111,432.650","52,420.050",3.146,160.797,0.008,Lift Training Leggings,leggings,True,38,56.963,9623,31,46,"2,577.600","1,332.380",1.211,67.832,0.003,Lift Training Leggings,leggings,0.099,-0.615,-0.507,-0.578
4,P0005,Contour High-Rise Leggings,leggings,False,671,40.862,199009,1532,2010,"80,946.890","44,123.690",2.996,120.636,0.008,Contour High-Rise Leggings,leggings,True,60,44.952,16725,107,147,"6,455.670","3,762.630",2.450,107.594,0.006,Contour High-Rise Leggings,leggings,0.100,-0.182,-0.130,-0.108
5,P0006,Pulse Pocket Leggings,leggings,False,689,45.407,158776,1179,1584,"70,730.580","33,981.780",2.299,102.657,0.007,Pulse Pocket Leggings,leggings,True,42,50.471,7988,17,22,"1,047.380",536.980,0.524,24.938,0.002,Pulse Pocket Leggings,leggings,0.112,-0.772,-0.530,-0.757
6,P0007,Apex Support Sports Bra,sports_bras,False,625,29.648,178748,1394,1870,"54,828.150","28,292.850",2.992,87.725,0.008,Apex Support Sports Bra,sports_bras,True,106,32.200,25743,141,184,"5,682.780","3,071.820",1.736,53.611,0.005,Apex Support Sports Bra,sports_bras,0.086,-0.420,-0.232,-0.389
7,P0008,Core Strappy Sports Bra,sports_bras,False,625,35.319,166055,1253,1658,"57,682.940","34,470.940",2.653,92.293,0.008,Core Strappy Sports Bra,sports_bras,True,106,38.012,26520,196,245,"9,092.160","5,662.160",2.311,85.775,0.007,Core Strappy Sports Bra,sports_bras,0.076,-0.129,-0.016,-0.071
8,P0009,Motion Longline Sports Bra,sports_bras,False,648,39.057,165166,1299,1735,"66,695.800","41,763.850",2.677,102.926,0.008,Motion Longline Sports Bra,sports_bras,True,83,41.630,19701,102,134,"5,383.950","3,458.370",1.614,64.867,0.005,Motion Longline Sports Bra,sports_bras,0.066,-0.397,-0.269,-0.370
9,P0010,Lift High-Support Bra,sports_bras,False,644,32.740,142526,1042,1405,"45,069.700","25,793.100",2.182,69.984,0.007,Lift High-Support Bra,sports_bras,True,87,35.235,17260,82,120,"4,118.300","2,471.900",1.379,47.337,0.005,Lift High-Support Bra,sports_bras,0.076,-0.368,-0.256,-0.324


**Interpretation:** Across products with both price-increase and normal days, average selling price is 7.1% higher on price-increase days; unit velocity is -15.8% different and revenue per product-day is -10.2% different. This is descriptive demand movement, not a causal elasticity estimate.

## Customer Churn by Major Segment


In [27]:
def churn_table(frame, group_col):
    return (
        frame.groupby(group_col, observed=False)
        .agg(
            customers=("customer_id", "count"),
            churn_rate=("churned", "mean"),
            avg_purchase_frequency=("purchase_frequency", "mean"),
            avg_prior_spending=("prior_spending", "mean"),
            avg_discount_percent=("average_discount_percent", "mean"),
        )
        .reset_index()
        .sort_values("churn_rate", ascending=False)
    )

region_churn = churn_table(customers, "customer_region")
channel_churn = churn_table(customers, "acquisition_channel")
tenure_churn = churn_table(customers, "tenure_bucket")
purchase_frequency_churn = churn_table(customers, "purchase_frequency_bucket")
discount_churn = churn_table(customers, "discount_usage_segment")
price_exposure_churn = churn_table(customers, "price_exposure_segment")
spending_churn = churn_table(customers.assign(prior_spending_quintile=customers["prior_spending_quintile"].astype(str)), "prior_spending_quintile")

display(region_churn)
display(channel_churn)
display(tenure_churn)
display(purchase_frequency_churn)
display(spending_churn)
display(discount_churn)
display(price_exposure_churn)

show_bar(region_churn, "customer_region", "churn_rate", "Churn Rate by Region", y_label="Churn rate")
show_bar(channel_churn, "acquisition_channel", "churn_rate", "Churn Rate by Acquisition Channel", y_label="Churn rate")
show_bar(tenure_churn, "tenure_bucket", "churn_rate", "Churn Rate by Customer Tenure", y_label="Churn rate")
show_bar(price_exposure_churn, "price_exposure_segment", "churn_rate", "Churn Rate by Price-Increase Exposure", y_label="Churn rate")

highest_region_churn = region_churn.iloc[0]
highest_channel_churn = channel_churn.iloc[0]
exposed_churn = price_exposure_churn.set_index("price_exposure_segment").loc["Exposed to price increase", "churn_rate"]
unexposed_churn = price_exposure_churn.set_index("price_exposure_segment").loc["Not exposed", "churn_rate"]
show_note(
    f"Overall churn is {customers['churned'].mean():.1%}. Churn is highest in {highest_region_churn['customer_region']} "
    f"({highest_region_churn['churn_rate']:.1%}) and highest for the {highest_channel_churn['acquisition_channel']} channel "
    f"({highest_channel_churn['churn_rate']:.1%}). Price-exposed customers show a descriptive churn gap of {(exposed_churn - unexposed_churn) * 100:.2f} percentage points."
)


,customer_region,customers,churn_rate,avg_purchase_frequency,avg_prior_spending,avg_discount_percent
0,asia_pacific,1218,0.085,4.037,218.250,7.734
1,europe,2341,0.076,4.157,221.344,8.213
3,rest_of_world,836,0.072,4.213,211.849,8.563
4,uk,4301,0.071,4.160,226.786,8.197
2,north_america,3804,0.069,4.159,228.518,8.142


,acquisition_channel,customers,churn_rate,avg_purchase_frequency,avg_prior_spending,avg_discount_percent
5,paid_social,3565,0.078,4.070,215.158,8.527
0,affiliate,957,0.074,3.967,219.735,8.753
4,organic_search,2184,0.074,4.169,231.776,7.635
3,influencer,2451,0.073,4.204,230.280,8.260
1,direct,2045,0.066,4.213,230.405,7.020
2,email,1298,0.065,4.280,220.859,9.229


,tenure_bucket,customers,churn_rate,avg_purchase_frequency,avg_prior_spending,avg_discount_percent
3,366-540 days,3025,0.080,3.606,258.625,8.458
4,541-730 days,3415,0.077,3.381,337.936,8.256
2,181-365 days,3301,0.071,4.190,176.019,8.087
1,91-180 days,1620,0.065,5.514,119.330,7.827
0,0-90 days,1139,0.059,5.851,83.447,7.792


,purchase_frequency_bucket,customers,churn_rate,avg_purchase_frequency,avg_prior_spending,avg_discount_percent
0,0-1,259,0.112,0.750,58.195,8.378
1,1-2,1679,0.086,1.525,109.052,8.165
2,2-3,2399,0.082,2.501,173.280,7.924
4,4-5,1644,0.081,4.472,294.740,8.362
3,3-4,2937,0.065,3.604,191.834,8.028
6,7-10,1382,0.063,8.009,297.470,8.360
5,5-7,1982,0.060,5.871,342.920,8.307
7,10+,218,0.050,11.408,243.965,8.265


,prior_spending_quintile,customers,churn_rate,avg_purchase_frequency,avg_prior_spending,avg_discount_percent
1,"(145.97, 224.842]",2495,0.079,4.091,184.067,8.072
0,"(14.389000000000001, 84.412]",2500,0.073,2.996,52.419,8.220
2,"(224.842, 343.452]",2500,0.073,4.448,277.506,8.181
4,"(84.412, 145.97]",2505,0.073,3.782,114.131,8.160
3,"(343.452, 1224.89]",2500,0.066,5.436,494.334,8.178


...

,customer_region,churn_rate
0,asia_pacific,0.085386
1,europe,0.076036
3,rest_of_world,0.071770
4,uk,0.071379
2,north_america,0.068612


,acquisition_channel,churn_rate
5,paid_social,0.078261
0,affiliate,0.074190
4,organic_search,0.073718
3,influencer,0.073439
1,direct,0.065526
2,email,0.065485


,tenure_bucket,churn_rate
3,366-540 days,0.080000
4,541-730 days,0.076720
2,181-365 days,0.070888
1,91-180 days,0.064815
0,0-90 days,0.058824


,price_exposure_segment,churn_rate
0,Exposed to price increase,0.088211
1,Not exposed,0.066637


**Interpretation:** Overall churn is 7.3%. Churn is highest in asia_pacific (8.5%) and highest for the paid_social channel (7.8%). Price-exposed customers show a descriptive churn gap of 2.16 percentage points.

## Regional and Acquisition-Channel Performance


In [28]:
customer_region_base = customers.groupby("customer_region").agg(
    customers=("customer_id", "count"),
    churn_rate=("churned", "mean"),
    avg_purchase_frequency=("purchase_frequency", "mean"),
)
transaction_region_base = transactions.groupby("customer_region").agg(
    revenue=("order_value", "sum"),
    gross_margin=("gross_margin", "sum"),
    transactions=("transaction_id", "count"),
    units_sold=("quantity", "sum"),
    average_order_value=("order_value", "mean"),
)
region_performance = customer_region_base.join(transaction_region_base).reset_index()
region_performance["revenue_per_customer"] = region_performance["revenue"] / region_performance["customers"]
region_performance["gross_margin_rate"] = region_performance["gross_margin"] / region_performance["revenue"]
region_performance["transactions_per_customer"] = region_performance["transactions"] / region_performance["customers"]
region_performance = region_performance.sort_values("revenue", ascending=False)

customer_channel_base = customers.groupby("acquisition_channel").agg(
    customers=("customer_id", "count"),
    churn_rate=("churned", "mean"),
    avg_purchase_frequency=("purchase_frequency", "mean"),
)
transaction_channel_base = transactions.groupby("acquisition_channel").agg(
    revenue=("order_value", "sum"),
    gross_margin=("gross_margin", "sum"),
    transactions=("transaction_id", "count"),
    units_sold=("quantity", "sum"),
    average_order_value=("order_value", "mean"),
)
channel_performance = customer_channel_base.join(transaction_channel_base).reset_index()
channel_performance["revenue_per_customer"] = channel_performance["revenue"] / channel_performance["customers"]
channel_performance["gross_margin_rate"] = channel_performance["gross_margin"] / channel_performance["revenue"]
channel_performance["transactions_per_customer"] = channel_performance["transactions"] / channel_performance["customers"]
channel_performance = channel_performance.sort_values("revenue", ascending=False)

display(region_performance)
display(channel_performance)
show_bar(region_performance, "customer_region", "revenue", "Revenue by Region", y_label="Revenue")
show_bar(region_performance.sort_values("revenue_per_customer", ascending=False), "customer_region", "revenue_per_customer", "Revenue per Customer by Region", y_label="Revenue per customer")
show_bar(channel_performance, "acquisition_channel", "revenue", "Revenue by Acquisition Channel", y_label="Revenue")
show_bar(channel_performance.sort_values("revenue_per_customer", ascending=False), "acquisition_channel", "revenue_per_customer", "Revenue per Customer by Acquisition Channel", y_label="Revenue per customer")

top_region = region_performance.iloc[0]
top_channel = channel_performance.iloc[0]
best_region_value = region_performance.sort_values("revenue_per_customer", ascending=False).iloc[0]
best_channel_value = channel_performance.sort_values("revenue_per_customer", ascending=False).iloc[0]
show_note(
    f"{top_region['customer_region']} contributes the most revenue by region at {format_currency(top_region['revenue'])}, "
    f"while {best_region_value['customer_region']} has the highest revenue per customer at ${best_region_value['revenue_per_customer']:,.2f}. "
    f"{top_channel['acquisition_channel']} contributes the most channel revenue, and {best_channel_value['acquisition_channel']} leads on revenue per customer."
)


,customer_region,customers,churn_rate,avg_purchase_frequency,revenue,gross_margin,transactions,units_sold,average_order_value,revenue_per_customer,gross_margin_rate,transactions_per_customer
4,uk,4301,0.071,4.160,"975,407.990","519,870.900",16947,22523,57.556,226.786,0.533,3.940
2,north_america,3804,0.069,4.159,"869,283.550","463,192.180",15043,20074,57.787,228.518,0.533,3.955
1,europe,2341,0.076,4.157,"518,166.570","276,343.450",8998,11976,57.587,221.344,0.533,3.844
0,asia_pacific,1218,0.085,4.037,"265,828.420","142,108.500",4678,6146,56.825,218.250,0.535,3.841
3,rest_of_world,836,0.072,4.213,"177,105.480","94,308.700",3103,4116,57.076,211.849,0.533,3.712


,acquisition_channel,customers,churn_rate,avg_purchase_frequency,revenue,gross_margin,transactions,units_sold,average_order_value,revenue_per_customer,gross_margin_rate,transactions_per_customer
5,paid_social,3565,0.078,4.070,"767,038.570","407,199.520",13458,17889,56.995,215.158,0.531,3.775
3,influencer,2451,0.073,4.204,"564,416.560","300,572.240",9784,13086,57.688,230.280,0.533,3.992
4,organic_search,2184,0.074,4.169,"506,198.120","271,785.030",8672,11518,58.372,231.776,0.537,3.971
1,direct,2045,0.066,4.213,"471,177.270","254,100.800",8095,10724,58.206,230.405,0.539,3.958
2,email,1298,0.065,4.280,"286,674.650","150,987.240",5070,6721,56.543,220.859,0.527,3.906
0,affiliate,957,0.074,3.967,"210,286.840","111,178.900",3690,4897,56.988,219.735,0.529,3.856


,customer_region,revenue
4,uk,975407.990000
2,north_america,869283.550000
1,europe,518166.570000
0,asia_pacific,265828.420000
3,rest_of_world,177105.480000


,customer_region,revenue_per_customer
2,north_america,228.518283
4,uk,226.786326
1,europe,221.344114
0,asia_pacific,218.249934
3,rest_of_world,211.848660


,acquisition_channel,revenue
5,paid_social,767038.570000
3,influencer,564416.560000
4,organic_search,506198.120000
1,direct,471177.270000
2,email,286674.650000
0,affiliate,210286.840000


,acquisition_channel,revenue_per_customer
4,organic_search,231.775696
1,direct,230.404533
3,influencer,230.280114
2,email,220.858744
0,affiliate,219.735465
5,paid_social,215.158084


**Interpretation:** uk contributes the most revenue by region at $975,408, while north_america has the highest revenue per customer at $228.52. paid_social contributes the most channel revenue, and organic_search leads on revenue per customer.

## Purchase Frequency and Customer Value Distribution


In [29]:
purchase_frequency_summary = customers["purchase_frequency"].describe(
    percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]
)
customer_value_summary = customers["total_spending"].describe(
    percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]
)

display(purchase_frequency_summary.to_frame("purchase_frequency"))
display(customer_value_summary.to_frame("total_spending"))

if PLOTLY_AVAILABLE:
    fig = px.histogram(customers, x="purchase_frequency", nbins=40, title="Purchase Frequency Distribution", template="plotly_white")
    fig.update_layout(xaxis_title="Annualized purchase frequency", yaxis_title="Customers")
    fig.show()
    fig = px.histogram(customers, x="total_spending", nbins=40, title="Customer Total Spending Distribution", template="plotly_white")
    fig.update_layout(xaxis_title="Total spending", yaxis_title="Customers")
    fig.show()
else:
    display(customers[["purchase_frequency", "total_spending"]].describe().T)

median_frequency = customers["purchase_frequency"].median()
median_spending = customers["total_spending"].median()
show_note(
    f"The median customer has {median_frequency:.2f} annualized purchases and ${median_spending:,.2f} in total spending, "
    "which gives later dashboard segments a practical baseline for customer value tiers."
)


,purchase_frequency
count,"12,500.000"
mean,4.151
std,2.234
min,0.502
5%,1.240
25%,2.509
50%,3.829
75%,5.345
95%,8.000
max,16.000


,total_spending
count,"12,500.000"
mean,224.463
std,167.139
min,14.390
5%,35.966
25%,98.980
50%,183.505
75%,304.952
95%,560.159
max,"1,224.890"


,count,mean,std,min,25%,50%,75%,max
purchase_frequency,"12,500.000",4.151,2.234,0.502,2.509,3.829,5.345,16.000
total_spending,"12,500.000",224.463,167.139,14.390,98.980,183.505,304.952,"1,224.890"


**Interpretation:** The median customer has 3.83 annualized purchases and $183.50 in total spending, which gives later dashboard segments a practical baseline for customer value tiers.

## Key Business Findings


In [30]:
overall_gmv = transactions["gross_merchandise_value"].sum()
overall_revenue = transactions["order_value"].sum()
overall_margin = transactions["gross_margin"].sum()
overall_churn_rate = customers["churned"].mean()
overall_aov = transactions["order_value"].mean()
transactions_per_customer = len(transactions) / len(customers)

peak_revenue_month = monthly_summary.loc[monthly_summary["revenue"].idxmax()]
leading_category = category_summary.iloc[0]
highest_margin_category = category_summary.sort_values("gross_margin_rate", ascending=False).iloc[0]
top_product = top_products_by_revenue.iloc[0]
top_margin_product = top_products_by_margin.iloc[0]
slowest_product = slowest_products.iloc[0]
highest_stockout_product = stockout_by_product.iloc[0]
highest_stockout_category = stockout_by_category.iloc[0]
highest_region_churn = region_churn.iloc[0]
highest_channel_churn = channel_churn.iloc[0]
top_region = region_performance.iloc[0]
top_channel = channel_performance.iloc[0]

price_churn_lookup = price_exposure_churn.set_index("price_exposure_segment")["churn_rate"]
price_exposed_churn = price_churn_lookup.get("Exposed to price increase", np.nan)
price_unexposed_churn = price_churn_lookup.get("Not exposed", np.nan)
price_churn_gap = price_exposed_churn - price_unexposed_churn

avg_price_lift = price_demand_comparison["price_lift_pct"].mean()
avg_velocity_change = price_demand_comparison["unit_velocity_change_pct"].mean()
avg_revenue_day_change = price_demand_comparison["revenue_per_day_change_pct"].mean()
add_to_cart_rate = funnel_rates.loc[funnel_rates["stage"] == "Add to cart", "rate"].iloc[0]
checkout_start_rate = funnel_rates.loc[funnel_rates["stage"] == "Checkout started", "rate"].iloc[0]
purchase_after_checkout_rate = funnel_rates.loc[
    funnel_rates["stage"] == "Purchase after checkout", "rate"
].iloc[0]

findings = [
    f"The synthetic business generated {format_currency(overall_gmv)} in GMV and {format_currency(overall_revenue)} in net revenue, with {format_currency(overall_margin)} in gross margin and a {overall_margin / overall_revenue:.1%} gross margin rate.",
    f"Monthly revenue peaks in {peak_revenue_month['transaction_month_label']} at {format_currency(peak_revenue_month['revenue'])}; average order value is ${overall_aov:,.2f} across {len(transactions):,} transactions.",
    f"The product funnel converts {add_to_cart_rate:.1%} of views to carts, {checkout_start_rate:.1%} of carts to checkout starts, and {purchase_after_checkout_rate:.1%} of checkout starts to purchases.",
    f"{leading_category['product_category']} is the largest revenue category at {format_currency(leading_category['revenue'])} ({leading_category['revenue_share']:.1%} of product-event revenue), while {highest_margin_category['product_category']} has the highest category gross margin rate at {highest_margin_category['gross_margin_rate']:.1%}.",
    f"{top_product['product_name']} is the top revenue product at {format_currency(top_product['revenue'])}; {top_margin_product['product_name']} contributes the most gross margin at {format_currency(top_margin_product['gross_margin'])}.",
    f"Product sales velocity ranges from {slowest_product['units_per_day']:.2f} to {fastest_products['units_per_day'].max():.2f} units per day, with {slowest_product['product_name']} among the slowest-moving products.",
    f"Product-day stockout frequency is {product_events['stockout_flag'].mean():.1%}; {highest_stockout_product['product_name']} has the highest product stockout rate at {highest_stockout_product['stockout_rate']:.1%}, and {highest_stockout_category['product_category']} is the highest-stockout category.",
    f"On price-increase product-days, average selling price is {avg_price_lift:.1%} higher for products with both normal and increased-price days, while unit velocity changes by {avg_velocity_change:.1%} and revenue per product-day changes by {avg_revenue_day_change:.1%}; this is descriptive, not causal.",
    f"Overall customer churn is {overall_churn_rate:.1%}; churn is highest in {highest_region_churn['customer_region']} ({highest_region_churn['churn_rate']:.1%}) and highest for the {highest_channel_churn['acquisition_channel']} channel ({highest_channel_churn['churn_rate']:.1%}).",
    f"Customers exposed to price increases have a {price_exposed_churn:.1%} churn rate versus {price_unexposed_churn:.1%} for unexposed customers, a descriptive gap of {price_churn_gap * 100:.2f} percentage points; {top_region['customer_region']} and {top_channel['acquisition_channel']} lead revenue by region and acquisition channel."
]

print("Key Business Findings")
print("=====================")
for index, finding in enumerate(findings, start=1):
    print(f"{index}. {finding}")


Key Business Findings
1. The synthetic business generated $3,075,245 in GMV and $2,805,792 in net revenue, with $1,495,824 in gross margin and a 53.3% gross margin rate.
2. Monthly revenue peaks in 2025-11 at $222,187; average order value is $57.53 across 48,769 transactions.
3. The product funnel converts 10.0% of views to carts, 40.0% of carts to checkout starts, and 18.0% of checkout starts to purchases.
4. leggings is the largest revenue category at $550,702 (19.6% of product-event revenue), while sports_bras has the highest category gross margin rate at 57.1%.
5. Lift Zip Jacket is the top revenue product at $119,532; Lift Zip Jacket contributes the most gross margin at $67,849.
6. Product sales velocity ranges from 1.41 to 3.05 units per day, with Apex Training Jacket among the slowest-moving products.
7. Product-day stockout frequency is 5.6%; Tempo Woven Joggers has the highest product stockout rate at 36.8%, and joggers is the highest-stockout category.
8. On price-increase pr